In [ ]:
import numpy as np
import random

# 第18章 多智能体强化学习

## 目录
1. 多智能体框架
2. 博弈论基础
3. 编程题

---
## 1. 多智能体框架

In [ ]:
"""多智能体强化学习"""
class MultiAgentEnv:
    """多智能体环境"""
    def __init__(self, n_agents=2):
        self.n_agents = n_agents
        self.states = [np.zeros(4) for _ in range(n_agents)]
    def reset(self):
        self.states = [np.random.randn(4) * 0.1 for _ in range(self.n_agents)]
        return self.states
    def step(self, actions):
        rewards = []
        for s, a in zip(self.states, actions):
            s += np.random.randn(4) * 0.05
            rewards.append(np.sum(s))
        self.states = [s + np.random.randn(4) * 0.05 for s in self.states]
        done = all(np.sum(s ** 2) > 5 for s in self.states)
        return self.states, rewards, done
    
def get_valid_actions(agent_id): return [0, 1]

print("多智能体: 合作/竞争/混合博弈")

---
## 2. 博弈论基础

In [ ]:
def nash_equilibrium(payoff_matrix):
    """求纳什均衡"""
    n_actions = len(payoff_matrix)
    for i, row in enumerate(payoff_matrix):
        for j, val in enumerate(row):
            if all(val >= payoff_matrix[k][j] for k in range(n_actions)):
                if all(val >= payoff_matrix[i][k] for k in range(n_actions)):
                    return (i, j)
    return None

def pure_strategy(payoff_matrix):
    """纯策略收益"""
    return payoff_matrix

def mixed_strategy(p):
    """混合策略"""
    probs = np.array(p)
    probs = probs / np.sum(probs)
    return probs

print("纳什均衡: 无智能体能单方面获益")

---
## 3. 编程题

### 编程题1：实现多智能体合作环境下的Q-Learning

In [ ]:
class IndependentQL:
    """独立Q学习"""
    def __init__(self, n_agents, n_states, n_actions):
        self.n_agents = n_agents
        self.q_tables = [np.zeros((n_states, n_actions)) for _ in range(n_agents)]
        self.gamma = 0.99
    
    def select_actions(self, states, epsilon=0.1):
        actions = []
        for i, s in enumerate(states):
            if random.random() < epsilon:
                actions.append(random.randint(0, 1))
            else:
                actions.append(np.argmax(self.q_tables[i][s]))
        return actions
    
    def update(self, agent_id, s, a, r, ns):
        self.q_tables[agent_id][s, a] += 0.1 * (r + self.gamma * np.max(self.q_tables[agent_id][ns]) - self.q_tables[agent_id][s, a])

class CooperativeEnv:
    """合作环境"""
    def __init__(self, n_agents=2):
        self.n_agents = n_agents
        self.state = 0
    def reset(self): self.state = 0; return [self.state] * self.n_agents
    def step(self, actions):
        if all(a == 1 for a in actions):
            self.state = min(4, self.state + 1)
        reward = 1.0 if self.state == 4 else 0.0
        done = self.state == 4
        return [self.state] * self.n_agents, [reward] * self.n_agents, done

def train_cooperative_q():
    """训练合作Q学习"""
    n_agents, n_states, n_actions = 2, 5, 2
    agents = IndependentQL(n_agents, n_states, n_actions)
    env = CooperativeEnv()
    rewards_history = []
    
    for ep in range(100):
        states = env.reset()
        total = 0
        
        for _ in range(20):
            actions = agents.select_actions(states)
            ns, rewards, done = env.step(actions)
            for i, (s, a, r) in enumerate(zip(states, actions, rewards)):
                agents.update(i, s, a, r, ns[i])
            states = ns
            total += sum(rewards)
            if done: break
        rewards_history.append(total)
    
    print("合作Q学习结果:")
    print(f"  最终回报: {rewards_history[-1]}")
    print(f"  收敛步数: {next((i for i, r in enumerate(rewards_history) if r > 0.9), -1)}")

train_cooperative_q()

---

### 编程题2：实现QMIX算法的简化版本

In [ ]:
class QMIX:
    """QMIX算法(简化)"""
    def __init__(self, n_agents, n_states, n_actions):
        self.n_agents = n_agents
        self.q_agent = [np.zeros((n_states, n_actions)) for _ in range(n_agents)]
        self.q_mixer = np.zeros((n_states, 1))
    
    def get_individual_q(self, agent_id, state, action):
        return self.q_agent[agent_id][state, action]
    
    def get_joint_q(self, states, actions):
        joint_q = sum(self.q_agent[i][s, a] for i, (s, a) in enumerate(zip(states, actions)))
        return joint_q * 0.5
    
    def update(self, states, actions, reward):
        joint_q = self.get_joint_q(states, actions)
        error = reward - joint_q
        for i, (s, a) in enumerate(zip(states, actions)):
            self.q_agent[i][s, a] += 0.1 * error

def test_qmix():
    """测试QMIX"""
    qmix = QMIX(n_agents=2, n_states=5, n_actions=2)
    env = CooperativeEnv()
    rewards = []
    
    for ep in range(50):
        s = env.reset()
        total = 0
        for _ in range(20):
            a = [random.randint(0, 1) for _ in range(2)]
            ns, r, done = env.step(a)
            qmix.update(s, a, sum(r))
            s = ns
            total += sum(r)
            if done: break
        rewards.append(total)
    
    print("QMIX测试:")
    print(f"  最终回报: {rewards[-1]}")

test_qmix()

---

### 编程题3：实现多智能体竞争环境下的学习

In [ ]:
class CompetitiveQL:
    """竞争环境Q学习"""
    def __init__(self, n_agents, n_states, n_actions):
        self.q_tables = [np.zeros((n_states, n_actions)) for _ in range(n_agents)]
    
    def select_actions(self, states, epsilon=0.1):
        return [np.argmax(q[s]) if random.random() > epsilon else random.randint(0, 1) 
                for q, s in zip(self.q_tables, states)]

class ZeroSumEnv:
    """零和博弈环境"""
    def __init__(self):
        self.state = np.random.randint(0, 2)
    def reset(self): self.state = np.random.randint(0, 2); return [self.state] * 2
    def step(self, actions):
        if actions[0] == actions[1]:
            rewards = [-1.0, 1.0]
        else:
            rewards = [1.0, -1.0]
        done = True
        return [self.state] * 2, rewards, done

def train_competitive():
    """训练零和博弈"""
    agents = CompetitiveQL(n_agents=2, n_states=2, n_actions=2)
    env = ZeroSumEnv()
    wins = [0, 0]
    
    for ep in range(100):
        states = env.reset()
        for _ in range(10):
            actions = agents.select_actions(states)
            ns, rewards, done = env.step(actions)
            for i, (s, a, r) in enumerate(zip(states, actions, rewards)):
                agents.q_tables[i][s, a] += 0.1 * r
            states = ns
            if any(r > 0 for r in rewards):
                wins = [w + (1 if r > 0 else 0) for w, r in zip(wins, rewards)]
            if done: break
    
    print("零和博弈结果:")
    print(f"  智能体1胜率: {wins[0]/ep:.1%}")
    print(f"  智能体2胜率: {wins[1]/ep:.1%}")

train_competitive()

print("\n第18章 多智能体强化学习 - 编程题完成!")